In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("Amazon_Review_Data-Copy1.xlsx")

In [3]:
df_clean = df.copy()

In [4]:
#EDA
df.head()

,Unique_ID,Category,Review_Header,Review_text,Rating,Own_Rating
0,136040,smartTv,Nice one,I liked it,5,Positive
1,134236,mobile,Huge battery life with amazing display,I bought the phone on Amazon and been using my...,5,Positive
2,113945,books,Four Stars,"Awesome book at reasonable price, must buy ......",4,Positive
3,168076,smartTv,Nice quality,good,5,Positive
4,157302,books,Nice book,"The book is fine,not bad,contains nice concept...",3,Neutral


In [5]:
df.isnull().sum() # tells us how many missing values are there

Unique_ID         0
Category          0
Review_Header     6
Review_text      37
Rating            0
Own_Rating        0
dtype: int64

In [6]:
df.describe()

,Unique_ID,Rating
count,60889.000000,60889.000000
mean,140444.000000,4.081148
std,17577.284607,1.342067
min,110000.000000,1.000000
25%,125222.000000,4.000000
50%,140444.000000,5.000000
75%,155666.000000,5.000000
max,170888.000000,5.000000


In [7]:
df["Rating"].value_counts().sort_index() 

Rating
1     6979
2     2108
3     4366
4    12976
5    34460
Name: count, dtype: int64

In [8]:
pd.crosstab(df["Rating"],df["Own_Rating"])# this tells how Own_Rating was assigned from rating

Own_Rating,Negative,Neutral,Positive
Rating,,,
1,6979,0,0
2,2108,0,0
3,0,4366,0
4,0,0,12976
5,0,0,34460


In [9]:
df["Review_text"].sample(10,random_state=42)

9998                                              Fabulous
31306                                         Good product
29066    Sound is average. Camera is pathetic, battery ...
18603                                            Very good
43932    Awesome book for self improvement, after readi...
11578                                            very good
43686    The phone which we bought is not turning on an...
57037                                                 Nice
39394    Used product past 15days and battery back is g...
29874                         Battery life is ðŸ‘ðŸ‘ðŸ‘
Name: Review_text, dtype: object

In [10]:
df["Review_text"].duplicated().sum() # tell how may duplicate reviews are there in my dataset

np.int64(11007)

In [11]:
df["Review_text"].str.len().describe()# compute statistics the no. of character

count    60847.000000
mean       141.667034
std        311.677814
min          1.000000
25%         18.000000
50%         61.000000
75%        158.000000
max      18294.000000
Name: Review_text, dtype: float64

In [12]:
df["Own_Rating"].value_counts()# how many negative pso nutra reviews we have 

Own_Rating
Positive    47436
Negative     9087
Neutral      4366
Name: count, dtype: int64

In [13]:
df["Category"].value_counts()

Category
mobile                22749
mobile accessories    14811
smartTv               14624
refrigerator           4791
books                  3914
Name: count, dtype: int64

In [14]:
df["Review_text"].isnull().sum()

np.int64(37)

In [15]:
df_clean = df_clean[df_clean["Review_text"].map(type) == str] # this basically returns only string vlaues in review_text

In [16]:
df_clean["Review_text"].map(type).value_counts()

Review_text
<class 'str'>    60847
Name: count, dtype: int64

In [17]:
# cleaning the data 
import re

df_clean["Review_text"] = df_clean["Review_text"].str.lower()
df_clean["Review_text"] = df_clean["Review_text"].str.replace(r'\s+', ' ', regex=True).str.strip()
df_clean["Review_text"] = df_clean["Review_text"].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', ' ', x))

In [18]:
df_clean["Review_text"].head(20)

print("cleaning applied successfully!")
print(df_clean["Review_text"].head(20))

cleaning applied successfully!
0                                            i liked it
1     i bought the phone on amazon and been using my...
2     awesome book at reasonable price  must buy    ...
3                                                  good
4     the book is fine not bad contains nice concept...
5     nice tv and pic quality  good custmer srrvice ...
6     the iphone 7 is legitimately among the most in...
7          20000 mah  what more you need  super product
8     the company should give more bettany backup an...
9                                       very good phone
10                                          good option
11    redmi note 6 pro is the best mobile at the bes...
12                                                 good
13    product is good as expected but after sale ser...
14                                    good product     
15                                           good phone
16    i   m a fan of alexa fire stick  its a value f...
17               

In [19]:
for col in df_clean.columns:
    if df_clean[col].apply(lambda x:isinstance(x,list)).any():
        print("List found in column:",col)

In [20]:
df_clean[df_clean["Review_text"].map(type) != str][["Review_text","Rating","Own_Rating"]] 
# show us the rows where Review_text is not a string

,Review_text,Rating,Own_Rating


In [21]:
print("Total rows:",len (df_clean))
print("duplicate review text:",df_clean["Review_text"].duplicated().sum())
print("missing review text:",df_clean["Review_text"].isna().sum())

Total rows: 60847
duplicate review text: 11724
missing review text: 0


In [22]:
df_clean.isnull().sum()

Unique_ID        0
Category         0
Review_Header    3
Review_text      0
Rating           0
Own_Rating       0
dtype: int64

In [23]:
print("Total rows:", len(df_clean))
print("Duplicate Review_text:", df_clean["Review_text"].duplicated().sum())
print("Missing Review_text:", df_clean["Review_text"].isna().sum())

Total rows: 60847
Duplicate Review_text: 11724
Missing Review_text: 0


In [24]:
def remove_duplicate_reviews(df_clean): # remove duplicates
    df_clean = df_clean.drop_duplicates(
        subset="Review_text",
        keep="first"
    ).reset_index(drop=True)
    return df_clean

df_clean = remove_duplicate_reviews(df_clean)

In [25]:
from sklearn.model_selection import train_test_split

X_temp, X_unseen, Y_temp, Y_unseen = train_test_split(
    df_clean["Review_text"],
    df_clean["Own_Rating"],
    test_size=0.005,
    random_state=42,
    stratify=df_clean["Own_Rating"]
)

X_train, X_val, Y_train, Y_val = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.19598,
    random_state=42,
    stratify=Y_temp
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Unseen:", len(X_unseen))

Training: 39298
Validation: 9579
Unseen: 246


In [26]:
# !pip install torch transformers

In [27]:
import torch
import transformers

print("torch ver:", torch.__version__)
print("transformer ver:", transformers.__version__)

torch ver: 2.14.0+cpu
transformer ver: 5.16.1


In [28]:
from transformers import AutoTokenizer # loading the tokennizer for DistillBERT

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)

print("Tokenizer Loaded Successfully!")

Tokenizer Loaded Successfully!


In [29]:
# Convert sentiment labels to numerical labels

label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

Y_train_encoded = Y_train.map(label_map)
Y_val_encoded = Y_val.map(label_map)
Y_unseen_encoded = Y_unseen.map(label_map)

print("Label mapping:", label_map)

print("\nTraining labels:")
print(Y_train_encoded.value_counts())

print("\nValidation labels:")
print(Y_val_encoded.value_counts())

print("\nUnseen labels:")
print(Y_unseen_encoded.value_counts())

Label mapping: {'Negative': 0, 'Neutral': 1, 'Positive': 2}

Training labels:
Own_Rating
2    29352
0     6899
1     3047
Name: count, dtype: int64

Validation labels:
Own_Rating
2    7155
0    1681
1     743
Name: count, dtype: int64

Unseen labels:
Own_Rating
2    184
0     43
1     19
Name: count, dtype: int64


In [30]:
# %pip install datasets

In [31]:
# %pip install --upgrade --force-reinstall pyarrow

In [32]:
from datasets import Dataset

# Create training dataset
train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": Y_train_encoded.tolist()
})

# Create validation dataset
val_dataset = Dataset.from_dict({
    "text": X_val.tolist(),
    "label": Y_val_encoded.tolist()
})

print("Training dataset:", train_dataset)
print("Validation dataset:", val_dataset)

Training dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 39298
})
Validation dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 9579
})


In [33]:
# converting amazon reviews text ready for distilbert
from transformers import AutoTokenizer

# Load the DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)

# Function to tokenize the text
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

# Tokenize training data
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

# Tokenize validation data
val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

print("Training data tokenized:", train_tokenized)
print("Validation data tokenized:", val_tokenized)

Map:   0%|          | 0/39298 [00:00<?, ? examples/s]

Map:   0%|          | 0/9579 [00:00<?, ? examples/s]

Training data tokenized: Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 39298
})
Validation data tokenized: Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 9579
})


In [34]:
# load pre trained distil bert model 
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3
)

print("DistilBERT model loaded successfully!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully!


In [35]:
# instructions for training/setting for training 

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_sentiment",

    # Training settings
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,

    # Evaluation
    eval_strategy="epoch",

    # Save model after each epoch
    save_strategy="epoch",
    save_total_limit=1,

    # Keep the best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=100,

    # CPU
    use_cpu=True,

    # Don't send anything to external tracking services
    report_to="none"
)

print("Training arguments created successfully!")

Training arguments created successfully!


In [36]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(
    "./distilbert_sentiment_final"
)

tokenizer = AutoTokenizer.from_pretrained(
    "./distilbert_sentiment_final"
)

print("Final DistilBERT model loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Final DistilBERT model loaded successfully!


In [37]:
# %pip install -U "Aaccelaerate>=1.1.0"

In [38]:
# %pip install -U accelerate --index-url https://pypi.org/simple

In [39]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
)
print("Trainer created successfully!")

Trainer created successfully!


In [40]:
# trainer.train(resume_from_checkpoint="./distilbert_sentiment/checkpoint-9825")

In [41]:
# trainer.save_model("./distilbert_sentiment_final")
# tokenizer.save_pretrained("./distilbert_sentiment_final")
# print("Final Distilbert model saved successfully!")

In [42]:
# eval_results =trainer.evaluate()
# print(eval_results)

In [43]:
# predictions=trainer.predict(val_tokenized)
# print("Prediction completed!")

In [44]:
# import numpy as np
# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     classification_report
# )

# y_pred = np.argmax(predictions.predictions, axis=1)
# y_true = predictions.label_ids

# accuracy = accuracy_score(y_true, y_pred)
# macro_precision = precision_score(y_true, y_pred, average="macro")
# macro_recall = recall_score(y_true, y_pred, average="macro")
# macro_f1 = f1_score(y_true, y_pred, average="macro")
# weighted_f1 = f1_score(y_true, y_pred, average="weighted")

In [45]:
# print("Accuracy:", accuracy)
# print("Macro Precision:", macro_precision)
# print("Macro Recall:", macro_recall)
# print("Macro F1:", macro_f1)
# print("Weighted F1:", weighted_f1)

# print("\nClassification Report:")
# print(
#     classification_report(
#         y_true,
#         y_pred,
#         target_names=["Negative", "Neutral", "Positive"]
#     )
# )

In [46]:
#pridiction function
import torch  
import torch.nn.functional as F

label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

def predict_sentiment(review):
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=-1)

    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class].item()

    sentiment = label_map[predicted_class]

    print("Review:", review)
    print("Sentiment:", sentiment)
    print("Confidence:", f"{confidence * 100:.2f}%")

In [47]:
predict_sentiment("the product is not good ")

Review: the product is not good 
Sentiment: Negative
Confidence: 98.40%
